# KI-gestütztes Klassifikationsmodell für Formel 1
![Formula 1 Car](../images/f1.jpg)


## 🎯 Ziel des Projekts

Ziel dieses Projekts ist es, ein **KI-gestütztes Klassifikationsmodell** zu entwickeln, das **vor einem Rennen** vorhersagt, ob ein Fahrer:

→ **unter die Top 10 kommt oder nicht**

Die Vorhersage basiert **ausschließlich auf vor dem Rennen bekannten Informationen**, um unrealistische oder verfälschende Vorhersagen (**Data Leakage**) zu vermeiden.

---

### 🔢 Formal:

**Eingabe:**  
*Pre-Race-Merkmale*:
- Startplatz  
- Erfahrung  
- Teamstärke  
- Strecke  
- Saison

**Ausgabe:**  
- `1` → Fahrer beendet das Rennen **in den Top 10**  
- `0` → Fahrer beendet das Rennen **außerhalb der Top 10**

---

## ❓ Warum ist dieses Problem für KI geeignet?

Es handelt sich um ein **überwachtes Lernproblem (Supervised Learning)**.

Es gibt:
- viele **historische Daten**
- eine **klare Zielvariable**
- **komplexe Zusammenhänge** zwischen Merkmalen


In [1]:
import kagglehub
import os

path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")

print("Downloaded to:", path)
print("Example files:", os.listdir(path)[:15])


Downloaded to: /Users/sofiiavelykokhatska/.cache/kagglehub/datasets/rohanrao/formula-1-world-championship-1950-2020/versions/24
Example files: ['circuits.csv', 'status.csv', 'lap_times.csv', 'sprint_results.csv', 'drivers.csv', 'races.csv', 'constructors.csv', 'constructor_standings.csv', 'qualifying.csv', 'driver_standings.csv', 'constructor_results.csv', 'pit_stops.csv', 'seasons.csv', 'results.csv']


## Datenbeschaffung

Um ein Machine-Learning-Modell zur Vorhersage von **Formel-1-Rennergebnissen** zu trainieren, wird ein **historischer, strukturierter Datensatz** benötigt, der sowohl **Rennergebnisse** als auch **vor dem Rennen bekannte Einflussfaktoren** enthält (z. B. Startposition, Fahrer, Team, Strecke).

### ✅ Anforderungen an die Daten:

Die Daten müssen:
- ausreichend **groß** sein  
- **mehrere Saisons** umfassen  
- **reproduzierbar beschaffbar** sein  
- eine **konsistente Struktur** aufweisen  

---

### 📦 Datenquelle: *Kaggle* & *Ergast Formula 1 Dataset*

- **Kaggle** ist eine etablierte Plattform für Data-Science-Projekte.  
  Viele Kaggle-Datensätze basieren auf offiziellen Quellen und sind für Forschungs- und Ausbildungszwecke geeignet.
  
- Der verwendete Datensatz basiert auf der **Ergast Formula 1 API**, einer **frei verfügbaren**, gut dokumentierten Datenquelle mit historischen Formel-1-Daten seit **1950**.

**Die Ergast-Datenbank enthält unter anderem:**
- 🗓️ Rennkalender und Strecken  
- 🧑‍✈️ Fahrer und Teams  
- ⏱️ Qualifying-Ergebnisse  
- 🏁 Rennergebnisse  
- ⌚ Rundenzeiten  
- 🛠️ Boxenstopps  

> Diese Daten sind **strukturiert**, **konsistent** und **zeitlich vollständig**.

---

## ⚙️ Implementierung der Datenbeschaffung im Projekt

### 📥 Verwendung der *Kaggle API*

Statt die Daten **manuell herunterzuladen**, wurde die **Kaggle API** (über das Python-Paket `kagglehub`) verwendet.

**Vorteile dieses Ansatzes:**
- 🔁 **Reproduzierbare** Datenerhebung  
- 🧹 **Klare Trennung** von Code und Daten  
- 🚫 **Keine manuelle Abhängigkeit** von lokalen Dateien  
- 🧑‍💻 **Professioneller ML-Workflow**

Beim Ausführen des Notebooks werden die Daten **automatisch heruntergeladen und lokal gespeichert**.


In [2]:
import shutil
import pathlib
import os

repo_root = pathlib.Path.cwd().parent
target_dir = repo_root / "data" / "raw"
target_dir.mkdir(parents=True, exist_ok=True)

copied = 0
for f in os.listdir(path):
    if f.endswith(".csv"):
        shutil.copy(pathlib.Path(path) / f, target_dir / f)
        copied += 1

print("Copied CSV files:", copied)
print("Now in data/raw:", sorted([p.name for p in target_dir.iterdir()])[:20])


Copied CSV files: 14
Now in data/raw: ['.gitkeep', '2025', 'circuits.csv', 'constructor_results.csv', 'constructor_standings.csv', 'constructors.csv', 'driver_standings.csv', 'drivers.csv', 'lap_times.csv', 'pit_stops.csv', 'qualifying.csv', 'races.csv', 'results.csv', 'seasons.csv', 'sprint_results.csv', 'status.csv']


## Kritische Auseinandersetzung mit dem Datensatz

### ✅ Stärken des Datensatzes

- 📅 Sehr großer **zeitlicher Umfang** (1950–heute)  
- 🗄️ Klare **relationale Struktur** (ähnlich zu relationalen Datenbanken)  
- 📊 **Reale**, nicht synthetische Daten  
- ⏱️ Gut geeignet für **zeitabhängige Analysen**  
- 🎯 **Hohe Relevanz** für das gewählte Problem

---

### ⚠️ Herausforderungen und Einschränkungen

- ⚖️ **Historische Regeländerungen** (z. B. Punktevergabe, Technik)  
- 📉 **Unvollständige oder weniger detaillierte Daten** in frühen Saisons  
- 🌦️ Keine **Wetterdaten** oder **Reifendaten** verfügbar  
- 🧠 Gewisse **Leistungsfaktoren** (z. B. Fahrstil) **nicht direkt messbar**

> Diese Einschränkungen wurden **bewusst akzeptiert** und teilweise durch **Feature Engineering** (z. B. *Era-Buckets*) **kompensiert**.
